# The tumor-context library — Phase C

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

Phase C makes the virtual-trial divergence view broadly useful: every supported tumor context (NSCLC, breast, CRC, HCC, melanoma) has its own **baseline** (y0, growth), its own **survival link** (tumor-specific Weibull-PH), and **>=2 eligible TGI models** so model-selection risk is measurable per context.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
contexts = ['NSCLC', 'breast', 'CRC', 'HCC', 'melanoma']
print('records', len(ds), '| subsystems', dict(ds.by_subsystem()))

In [ ]:
# Each context: own baseline + survival link + >=2 eligible TGI models, measurable divergence.
t = np.linspace(0, 156, 313)
for tt in contexts:
    cmp = onkos.compare(ds, purpose='tgi', context={'tumor_type': tt, 'line': 'first'}, drug_effect=1.0, t=t)
    models = [tr.record_id for tr in cmp.included]
    rng = cmp.median_os_range
    print(f'{tt:9s} n={len(cmp.included)} divergence={cmp.os_divergence:.3f} '
          f'medOS_range={tuple(round(m,1) for m in rng) if rng else None}')
    assert len(cmp.included) >= 2 and cmp.os_divergence > 0

In [ ]:
# Population OS envelope (model-choice spread) per context.
for tt in contexts:
    cmp = onkos.compare(ds, purpose='tgi', context={'tumor_type': tt, 'line': 'first'}, drug_effect=1.0, t=t)
    curves = np.vstack([tr.os_curve for tr in cmp.included])
    plt.fill_between(t, curves.min(0), curves.max(0), alpha=0.25)
    plt.plot(t, curves.mean(0), lw=1.5, label=tt)
plt.axhline(0.5, ls=':', color='grey'); plt.ylim(0, 1.02)
plt.xlabel('weeks'); plt.ylabel('survival fraction'); plt.legend(fontsize=8); plt.title('OS by tumor context');

In [ ]:
# Transportability still bites: a CRC model applied to NSCLC is greyed out (floored to D).
cmp = onkos.compare(ds, purpose='tgi', context={'tumor_type': 'NSCLC', 'line': 'first'})
excluded = {rid for rid, _ in cmp.excluded}
print('excluded for NSCLC:', sorted(excluded))
assert any('crc' in e or 'breast' in e or 'hcc' in e or 'melanoma' in e for e in excluded)